In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import time
import json
import psycopg
import os
import sys
import httpx

sys.path.append(os.path.abspath('./src'))
from dota_db import DotaDB

db = DotaDB()

In [4]:
with httpx.Client() as client:
    patches = pd.DataFrame(db.fetch_opendota(client, endpoint='constants/patch'))
    db.create_table_from_df(patches, 'patches_opendota')
    db.insert_df_into_table(patches, 'patches_opendota')

In [10]:
with httpx.Client() as client:
    results = db.fetch_opendota(client, endpoint='constants/items')

In [33]:
items = []
for short_name, item in results.items():
    items.append(
        {
            'id': item['id'],
            'shortName': short_name,
            'displayName': item.get('dname'),
            'qual': item.get('qual'),
            'cost': item['cost'],
            'behavior': item.get('behavior'),
            'attributes': item['attrib'],
            'components': item.get('components'),
            'charges': item.get('charges'),
            'created': item['created']
        }
    )
items_df = pd.DataFrame(items)
db.create_table_from_df(items_df, 'item_details_opendota', jsonb_cols=['attributes'])
db.insert_df_into_table(items_df, 'item_details_opendota', jsonb_cols=['attributes'])

In [3]:
# Get game versions
query = """
    query {
        constants {
            gameVersions {
                id
                name
                asOfDateTime
            }
        }
    }
"""
with httpx.Client(headers=db.stratz_headers) as client:
    result = db.fetch_stratz(client, query)
df = pd.DataFrame(result['data']['constants']['gameVersions'])
df['asOfDateTime'] = df['asOfDateTime'].apply(datetime.fromtimestamp)
db.create_table_from_df(df, 'patches')
db.insert_df_into_table(df, 'patches')

In [ ]:
## Get npc data from stratz
query = """
    query($gameVersionId: Short!) {
        constants {
            npcs(gameVersionId: $gameVersionId) {
                id
                name
                stat {
                    statusHealth
                    statusHealthRegen
                    attackDamageMin
                    attackDamageMax
                    attackRate
                    attackRange
                    movementSpeed
                    isNeutralUnitType
                    isAncient
                    teamName
                }
            }
        }
    }
"""
variables = {'gameVersionId': 182}
result = db.fetch_stratz(query, variables=variables)
df = pd.DataFrame(result['data']['constants']['npcs'])
stats_df = pd.json_normalize(df['stat'])
df = df.join(stats_df).drop(columns=['stat'])
discard_patterns = [
    'thinker', 'companion', 'visual', 'sound', 'event', 
    'shmup', 'banana', 'target_dummy', 'looping', 'promo'
]
discard_regex = '|'.join(discard_patterns)
df_filtered = df[~df['name'].str.contains(discard_regex, case=False, na=False)]
df_filtered.dtypes
df_filtered.convert_dtypes(convert_integer=False).dtypes
db.create_table_from_df(df_filtered, 'npcs', False)
db.insert_df_into_table(df_filtered, 'npcs')

In [2]:
query = """
    query($gameVersionId: Short!) {
        constants {
            items(gameVersionId: $gameVersionId) {
                id
                        shortName
                displayName
                isSupportFullItem
                attributes {
                    name
                    value
                }
                stat {
                    cost
                    isRecipe
                    isSupport
                    behavior
                    manaCost
                    shopTags
                    needsComponents
                    itemResult
                    quality
                }
                components {
                    componentId
                }
            }
        }
    }
"""
variables = {'gameVersionId': 182}
result = db.fetch_stratz(query, variables=variables)

In [9]:
result_json = result['data']['constants']['items']
df = pd.json_normalize(result_json)
df = df.drop(['stat'], axis=1)
rename_dict = {}
for col_name in df.columns:
    if col_name.startswith('stat.'):
        rename_dict[col_name] = col_name[5:]
df = df.rename(rename_dict, axis=1)
## I used non-na components because isRecipe includes old
## recipes for which items are not active in current patch
df_recipes = df[df['components'].notna()]
df['behavior'] = df['behavior'].astype('Int64')
df['cost'] = df['cost'].astype('Int64')
df['itemResult'] = df['itemResult'].astype('Int64')

In [10]:
ids = []
shop_tags = []
for idx, row in df.iterrows():
    ids.append(row['id'])
    try:
        shop_tags.append(row['shopTags'].split(';'))
    except:
        shop_tags.append(row['shopTags'])
df = df.drop('shopTags', axis=1)
df_attributes_jsonb = df.copy()
for idx, row in df.copy().iterrows():
    flat_attrs = {}
    if isinstance(row['attributes'], list):
        for attr in row['attributes']:
            flat_attrs[attr['name']] = attr['value']
            df_attributes_jsonb.at[idx, 'attributes'] = flat_attrs
    else:
        df_attributes_jsonb.at[idx, 'attributes'] = pd.NA

In [11]:
rename_dict = {}
for col_name in df_attributes_jsonb.columns:
    new_colname = col_name
    for idx, char in enumerate(col_name):
        if str.isupper(char):
            new_colname = new_colname[: idx] + '_' + new_colname[idx:]
    new_colname = new_colname.lower()
    rename_dict[col_name] = new_colname
rename_dict['isSupportFullItem'] = 'is_support_full_item'
df_attributes_jsonb.rename(rename_dict, axis=1)

,id,short_name,display_name,is_support_full_item,attributes,components,cost,is_recipe,is_support,behavior,mana_cost,needs_components,item_result,quality
0,1,blink,Blink Dagger,None,"{'blink_damage_cooldown': '3.0', 'blink_range'...",None,2250,False,False,137439478800,[0],False,<NA>,component
1,2,blades_of_attack,Blades of Attack,None,{'bonus_damage': '9'},None,450,False,False,2,None,False,<NA>,component
2,3,broadsword,Broadsword,None,{'bonus_damage': '15'},None,1000,False,False,2,None,False,<NA>,component
3,4,chainmail,Chainmail,None,{'bonus_armor': '4'},None,550,False,False,2,None,False,<NA>,component
4,5,claymore,Claymore,None,{'bonus_damage': '20'},None,1350,False,False,2,None,False,<NA>,component
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
570,4207,recipe_great_famango,,None,<NA>,"[{'componentId': 4204}, {'componentId': 4204},...",0,True,False,0,None,False,4205,None
571,4208,recipe_greater_famango,,None,<NA>,"[{'componentId': 4205}, {'componentId': 4205}]",0,True,False,0,None,False,4206,None
572,4300,ofrenda,Beloved Memory,None,{'speed': '1000'},None,0,False,False,72,[0],False,<NA>,None
573,4301,ofrenda_shovel,Scrying Shovel,None,<NA>,None,0,False,False,134217936,[0],False,<NA>,None


In [21]:
db.create_table_from_df(df_attributes_jsonb, 'item_details', convert_dtypes=False, jsonb_cols=['attributes', 'components'])

Table 'item_details' created successfully.


In [19]:
mask = df_attributes_jsonb.apply(lambda col: col.astype(str).str.contains('\x00', na=False))
rows_with_nul = df_attributes_jsonb[mask.any(axis=1)]

Empty DataFrame
Columns: [id, shortName, displayName, isSupportFullItem, attributes, components, cost, isRecipe, isSupport, behavior, manaCost, needsComponents, itemResult, quality]
Index: []


In [18]:
## Since these are not active items currently (one is no longer, other not yet possibly)
## rows are simply dropped
df_attributes_jsonb = df_attributes_jsonb.drop([416, 430], axis=0)

In [22]:
db.insert_df_into_table(df_attributes_jsonb, 'item_details', jsonb_cols=['attributes', 'components'])

Data inserted into table 'item_details' successfully.


In [ ]:
query = '''
    query($gameVersionId: Short) {
        constants {
            heroes(gameVersionId: $gameVersionId) {
                id
                name
                displayName
                shortName
                gameVersionId
                roles {
                    roleId
                    level
                }
                abilities {
                    slot
                    gameVersionId
                    abilityId
                }
                stats {
                    startingArmor
                    startingMagicArmor
                    startingDamageMin
                    startingDamageMax
                    attackRange
                    attackType
                    moveSpeed
                    hpRegen
                    mpRegen
                    primaryAttribute
                    strengthGain
                    agilityGain
                    intelligenceGain
                    strengthBase
                    agilityBase
                    intelligenceBase
                }
                talents {
                    abilityId
                    slot
                }
                facets {
                    abilityId
                    facetId
                    slot
                }
            }
        }
    }
'''
with httpx.Client(headers=db.stratz_headers) as client:
    result = db.fetch_stratz(client, query, variables={'gameVersionId': 182})
    res = result['data']['constants']['heroes']

In [ ]:
df_hero_details = pd.DataFrame(res)
df_hero_abilities = pd.DataFrame(
    columns=[
        'heroId',
        'slot',
        'gameVersionId',
        'abilityId'
    ]
)
df_hero_talents = pd.DataFrame(
    columns=[
        'heroId',
        'abilityId',
        'slot'
    ]
)
df_hero_facets = pd.DataFrame(
    columns=[
        'heroId',
        'abilityId',
        'facetId',
        'slot'
    ]
)
for idx, row in df_hero_details.iterrows():
    df_ha = pd.DataFrame(row['abilities'])
    df_ha.insert(0, 'heroId', row['id'])
    df_hero_abilities = pd.concat([df_hero_abilities, df_ha])
    df_ht = pd.DataFrame(row['talents'])
    df_ht.insert(0, 'heroId', row['id'])
    df_hero_talents = pd.concat([df_hero_talents, df_ht])
    df_hf = pd.DataFrame(row['facets'])
    df_hf.insert(0, 'heroId', row['id'])
    df_hero_facets = pd.concat([df_hero_facets, df_hf])
df_hero_details = df_hero_details.drop(['abilities', 'talents', 'facets'], axis=1)
db.create_table_from_df(df_hero_details, 'hero_details', convert_dtypes=False, jsonb_cols=['roles', 'stats'])
db.insert_df_into_table(df_hero_details, 'hero_details', jsonb_cols=['roles', 'stats'])
db.create_table_from_df(df_hero_abilities, 'hero_abilities', add_serial_id=True)
db.insert_df_into_table(df_hero_abilities, 'hero_abilities')
db.create_table_from_df(df_hero_talents, 'hero_talents', add_serial_id=True)
db.insert_df_into_table(df_hero_talents, 'hero_talents')
db.create_table_from_df(df_hero_facets, 'hero_facets', add_serial_id=True)
db.insert_df_into_table(df_hero_facets, 'hero_facets')

C:\Users\benib\AppData\Local\Temp\ipykernel_50952\76676605.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_hero_facets = pd.concat([df_hero_facets, df_hf])
C:\Users\benib\AppData\Local\Temp\ipykernel_50952\76676605.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_hero_facets = pd.concat([df_hero_facets, df_hf])
C:\Users\benib\AppData\Local\Temp\ipykernel_50952\76676605.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a fu

Table 'hero_details' created successfully.
Data inserted into table 'hero_details' successfully.
Table 'hero_abilities' created successfully.
Data inserted into table 'hero_abilities' successfully.
Table 'hero_talents' created successfully.
Data inserted into table 'hero_talents' successfully.
Table 'hero_facets' created successfully.
Data inserted into table 'hero_facets' successfully.


In [48]:
query = '''
    query {
        constants {
            abilities(gameVersionId: 182) {
            id
            name
            uri
            language {
                displayName
                description
                aghanimDescription
                shardDescription
            }
            stat {
                abilityId
                type
                behavior
                unitDamageType
                unitTargetType
                unitTargetTeam
                unitTargetFlags
                duration
                damage
                castPoint
                castRange
                channelTime
                manaCost
                cooldown
                isGrantedByScepter
                isGrantedByShard
                hasScepterUpgrade
                hasShardUpgrade
                dispellable
                isInnate
                isUltimate
                linkedAbilityId
            }
            attributes {
                name
                value
                linkedSpecialBonusAbilityId
                requiresScepter
            }
            isTalent
        }
    }
}
'''
with httpx.Client(headers=db.stratz_headers) as client:
    result = db.fetch_stratz(client, query)
    res = result['data']['constants']['abilities']

In [113]:
def clean_lists(val):
    if isinstance(val, list):
        return ", ".join(map(str, val))
    return val

df_abilities = pd.DataFrame(res)
abilities_language = pd.json_normalize(df_abilities['language'])
new_cols = ['ability_' + colname for colname in abilities_language.columns]
rename_dict = {old_col: new_col for old_col, new_col in zip(abilities_language.columns, new_cols)}
abilities_language = abilities_language.rename(rename_dict, axis=1)
df_abilities = pd.concat([df_abilities, abilities_language], axis=1).drop('language', axis=1)
ability_stats = pd.json_normalize(df_abilities['stat']).drop('abilityId', axis=1)
df_abilities = pd.concat([df_abilities, ability_stats], axis=1).drop(['stat', 'attributes'], axis=1)
list_cols = ['duration', 'damage', 'castPoint', 'castRange', 'channelTime', 'manaCost', 'cooldown']
for col in list_cols:
    df_abilities[col] = df_abilities[col].apply(clean_lists)
db.create_table_from_df(df_abilities, 'ability_details')
db.insert_df_into_table(df_abilities, 'ability_details')

Table 'ability_details' created successfully.
Data inserted into table 'ability_details' successfully.


In [91]:
df_ability_attributes = pd.DataFrame(
    columns=[
        'abilityId',
        'name',
        'value',
        'linkedSpecialBonusAbilityId',
        'requiresScepter'
    ]
)
for data in res:
    df_aa = pd.DataFrame(data['attributes'])
    df_aa.insert(0, 'abilityId', data['id'])
    df_ability_attributes = pd.concat([df_ability_attributes, df_aa])

C:\Users\benib\AppData\Local\Temp\ipykernel_50952\971139022.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_ability_attributes = pd.concat([df_ability_attributes, df_aa])
C:\Users\benib\AppData\Local\Temp\ipykernel_50952\971139022.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_ability_attributes = pd.concat([df_ability_attributes, df_aa])
C:\Users\benib\AppData\Local\Temp\ipykernel_50952\971139022.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA

In [9]:
query = '''
    query($tiers: [LeagueTier], $betweenStartDateTime: Long, $betweenEndDateTime: Long) {
    leagues(request: {tiers: $tiers, betweenStartDateTime: $betweenStartDateTime, betweenEndDateTime: $betweenEndDateTime, take: 10000}) {
        id
        displayName
        tournamentUrl
        private
        freeToSpectate
        tier
        region
        prizePool
        basePrizePool
        startDateTime
        endDateTime
        lastMatchDate
        country
        venue
        nodeGroups {
        id
        nodeGroupType
        secondaryAdvancingTeamCount
        secondaryAdvancingNodeGroupId
        tertiaryAdvancingTeamCount
        tertiaryAdvancingNodeGroupId
        isFinalGroup
        isTieBreaker
        eliminationDPCPoints
        round
        maxRounds
        teamCount
        advancingTeamCount
        advancingNodeGroupId
        }
    }
}
'''
table_created = False
db_query = 'SELECT DISTINCT id FROM league_details;'
saved_leagues = [result[0] for result in db.select(db_query)]
with httpx.Client(headers=db.stratz_headers) as client:
    for tier in ['INTERNATIONAL', 'PROFESSIONAL']:
        results = db.fetch_stratz(
            client, 
            query, 
            variables={
                'tiers': [tier], 'betweenStartDateTime': int(datetime(2020, 1, 1).timestamp()),
                'betweenEndDateTime': int(datetime.now().timestamp())
            }
        )
        df_leagues = pd.DataFrame(results['data']['leagues'])
        print(len(df_leagues))
        for idx, row in df_leagues.iterrows():
            if len(row['nodeGroups']) != 0:
                df_node_groups = pd.json_normalize(row['nodeGroups'])
                df_node_groups.insert(1, 'league_id', row['id'])
                df_node_groups = df_node_groups.rename({'id': 'league_node_id'}, axis=1)
                if not table_created:
                    db.create_table_from_df(df_node_groups, 'league_node_groups', convert_dtypes=False, add_serial_id=True)
                    table_created = True
                db.insert_df_into_table(df_node_groups, 'league_node_groups')
        df_leagues = df_leagues.drop('nodeGroups', axis=1)
        if tier == 'INTERNATIONAL':
            db.create_table_from_df(df_leagues, 'league_details', convert_dtypes=False)
        for idx, row in df_leagues.copy().iterrows():
            if row['id'] in saved_leagues:
                df_leagues = df_leagues.drop(idx, axis=0)
        db.insert_df_into_table(df_leagues, 'league_details'r)

6
Table 'league_node_groups' created successfully.
Error inserting data into table 'league_node_groups': duplicate key value violates unique constraint "league_node_groups_pkey"
DETAIL:  Key (id)=(0) already exists.
CONTEXT:  COPY league_node_groups, line 1
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_node_groups' successfully.
Table 'league_details' created successfully.
Data inserted into table 'league_details' successfully.
1231
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_node_groups' successfully.
Data inserted into table 'league_nod